In [2]:
import numpy  as np
import pandas as pd 
from scipy import stats 
import matplotlib.pyplot as plt  

In [3]:
np.random.seed(42)

n_samples = 5000

In [4]:
train_amounts = np.random.normal(loc=100, scale=30, size=n_samples)
train_amounts = np.clip(train_amounts, 10, None)

In [5]:
train_preds = np.concatenate([
    np.random.normal(0.1, 0.05, int(n_samples * 0.85)),
    np.random.normal(0.9, 0.05, int(n_samples * 0.15))
])
train_preds = np.clip(train_preds, 0, 1)

reference_df = pd.DataFrame({
    'transaction_amount': train_amounts,
    'prediction_score': train_preds
})

In [6]:
prod_amounts = np.random.normal(loc=140, scale=45, size=n_samples) 
prod_amounts = np.clip(prod_amounts, 10, None)

In [7]:
prod_preds = np.concatenate([
    np.random.normal(0.3, 0.15, int(n_samples * 0.70)),
    np.random.normal(0.7, 0.15, int(n_samples * 0.30))
])
prod_preds = np.clip(prod_preds, 0, 1)

production_df = pd.DataFrame({
    'transaction_amount': prod_amounts,
    'prediction_score': prod_preds
})

# KS DRIFT MONITOR 

In [8]:
def run_ks_monitor(reference: np.ndarray , production: np.ndarray, feature_name: str):
    
    ks_stat , p_value = stats.ks_2samp(reference,production)
    
    is_drifting = p_value < 0.05
    status = "Drift Detected" if is_drifting else "Stable"
    
    print(f"--- {feature_name.upper()} ---")
    print(f"Status:       {status}")
    print(f"KS Statistic: {ks_stat:.4f} (Max distance between CDFs)")
    print(f"P-Value:      {p_value:.4e} (Probability of same distribution)")
    print("-" * 40 + "\n")

In [9]:
print("=== AUTOMATED DRIFT ALERTING SYSTEM ===\n")

run_ks_monitor(
    reference_df['transaction_amount'].values, 
    production_df['transaction_amount'].values, 
    "Input Feature: Transaction Amount"
)

run_ks_monitor(
    reference_df['prediction_score'].values, 
    production_df['prediction_score'].values, 
    "Model Output: Prediction Score"
)

=== AUTOMATED DRIFT ALERTING SYSTEM ===

--- INPUT FEATURE: TRANSACTION AMOUNT ---
Status:       Drift Detected
KS Statistic: 0.4372 (Max distance between CDFs)
P-Value:      0.0000e+00 (Probability of same distribution)
----------------------------------------

--- MODEL OUTPUT: PREDICTION SCORE ---
Status:       Drift Detected
KS Statistic: 0.6592 (Max distance between CDFs)
P-Value:      0.0000e+00 (Probability of same distribution)
----------------------------------------



In [11]:
def calculate_psi(expected: np.ndarray, actual: np.ndarray, buckets: int = 10) -> float:
    """
    Calculates the Population Stability Index (PSI) from scratch using decile binning.
    """
    percentiles = np.linspace(0, 100, buckets + 1)
    bin_edges = np.percentile(expected, percentiles)
    
    bin_edges = np.unique(bin_edges)
    
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    expected_fractions = expected_counts / len(expected)
    actual_fractions = actual_counts / len(actual)
    
    eps = 1e-4
    expected_fractions = np.where(expected_fractions == 0, eps, expected_fractions)
    actual_fractions = np.where(actual_fractions == 0, eps, actual_fractions)
    
    psi_values = (actual_fractions - expected_fractions) * np.log(actual_fractions / expected_fractions)
    total_psi = np.sum(psi_values)
    
    return total_psi

In [12]:

def evaluate_psi(psi_value: float, feature_name: str):
    """Interprets PSI against production thresholds."""
    if psi_value < 0.1:
        status = "🟢 STABLE (<0.1)"
    elif psi_value < 0.2:
        status = "🟡 MODERATE DRIFT (0.1 - 0.2)"
    else:
        status = "🔴 SEVERE DRIFT (>0.2)"
        
    print(f"--- {feature_name.upper()} ---")
    print(f"PSI Score: {psi_value:.4f}")
    print(f"Status:    {status}")
    print("-" * 40 + "\n")

In [13]:
print("=== PSI MAGNITUDE EVALUATION ===\n")

# Evaluate Input Features
psi_inputs = calculate_psi(
    reference_df['transaction_amount'].values, 
    production_df['transaction_amount'].values
)
evaluate_psi(psi_inputs, "Input Feature: Transaction Amount")

# Evaluate Output Predictions
psi_preds = calculate_psi(
    reference_df['prediction_score'].values, 
    production_df['prediction_score'].values
)
evaluate_psi(psi_preds, "Model Output: Prediction Score")

=== PSI MAGNITUDE EVALUATION ===

--- INPUT FEATURE: TRANSACTION AMOUNT ---
PSI Score: 0.9820
Status:    🔴 SEVERE DRIFT (>0.2)
----------------------------------------

--- MODEL OUTPUT: PREDICTION SCORE ---
PSI Score: 2.8491
Status:    🔴 SEVERE DRIFT (>0.2)
----------------------------------------



In [16]:
# JENSEN-SHANNON DIVERGENCE (JSD) MONITOR

def calculate_js_divergence(p_data: np.ndarray, q_data: np.ndarray, bins: int = 30) -> float:
    """
    Calculates the Jensen-Shannon Divergence bounded between 0 and 1.
    """
    min_val = min(np.min(p_data), np.min(q_data))
    max_val = max(np.max(p_data), np.max(q_data))
    bin_edges = np.linspace(min_val, max_val, bins + 1)
    
    p_counts, _ = np.histogram(p_data, bins=bin_edges)
    q_counts, _ = np.histogram(q_data, bins=bin_edges)
    
    eps = 1e-10
    P = (p_counts + eps) / np.sum(p_counts + eps)
    Q = (q_counts + eps) / np.sum(q_counts + eps)
    
    # 4. Calculate the Mixture Distribution (M)
    M = 0.5 * (P + Q)
    
    # 5. Calculate KL Divergence to the Mixture (using Base 2 log for 0-1 bounds)
    kl_pm = np.sum(P * np.log2(P / M))
    kl_qm = np.sum(Q * np.log2(Q / M))
    
    # 6. JS Divergence is the average of the two KL Divergences
    jsd = 0.5 * kl_pm + 0.5 * kl_qm
    
    return jsd

In [18]:

def evaluate_jsd(jsd_value: float, feature_name: str):
    """Interprets JS Divergence magnitude."""
    if jsd_value < 0.05:
        status = "🟢 STABLE (<0.05)"
    elif jsd_value < 0.1:
        status = "🟡 MODERATE DRIFT (0.05 - 0.1)"
    else:
        status = "🔴 SEVERE DRIFT (>0.1)"
        
    print(f"--- {feature_name.upper()} ---")
    print(f"JS Divergence: {jsd_value:.4f} (Scale: 0 to 1)")
    print(f"Status:        {status}")
    print("-" * 40 + "\n")

In [19]:

# Evaluate Input Features
jsd_inputs = calculate_js_divergence(
    reference_df['transaction_amount'].values, 
    production_df['transaction_amount'].values
)
evaluate_jsd(jsd_inputs, "Input Feature: Transaction Amount")

# Evaluate Output Predictions
jsd_preds = calculate_js_divergence(
    reference_df['prediction_score'].values, 
    production_df['prediction_score'].values
)
evaluate_jsd(jsd_preds, "Model Output: Prediction Score")

--- INPUT FEATURE: TRANSACTION AMOUNT ---
JS Divergence: 0.2115 (Scale: 0 to 1)
Status:        🔴 SEVERE DRIFT (>0.1)
----------------------------------------

--- MODEL OUTPUT: PREDICTION SCORE ---
JS Divergence: 0.5506 (Scale: 0 to 1)
Status:        🔴 SEVERE DRIFT (>0.1)
----------------------------------------



In [20]:
#  UNIFIED ML OBSERVABILITY REPORT
def generate_drift_report(reference_df: pd.DataFrame, production_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compiles KS, PSI, and JSD metrics into a unified summary table and triggers alerts.
    """
    report_data = []
    system_status = "HEALTHY"
    
    features_to_monitor = [
        ('transaction_amount', 'Input Feature'),
        ('prediction_score', 'Model Output')
    ]
    
    for col_name, feature_type in features_to_monitor:
        ref_data = reference_df[col_name].values
        prod_data = production_df[col_name].values
        
        # 1. Compute Metrics
        ks_stat, p_value = stats.ks_2samp(ref_data, prod_data)
        psi_val = calculate_psi(ref_data, prod_data)
        jsd_val = calculate_js_divergence(ref_data, prod_data)
        
        # 2. Evaluate Severe Drift Thresholds
        ks_severe = p_value < 0.05
        psi_severe = psi_val >= 0.20
        jsd_severe = jsd_val >= 0.10
        
        is_failing = ks_severe or psi_severe or jsd_severe
        if is_failing:
            system_status = "CRITICAL ALERT"
            
        report_data.append({
            "Feature/Output": f"{col_name} ({feature_type})",
            "KS Stat": round(ks_stat, 4),
            "KS p-value": f"{p_value:.2e}",
            "PSI": round(psi_val, 4),
            "JS Divergence": round(jsd_val, 4),
            "Status": "🚨 SEVERE DRIFT" if is_failing else "✅ STABLE"
        })
        
    report_df = pd.DataFrame(report_data)
    
    # 3. Render Dashboard Output
    print("=" * 65)
    print(f"      SYSTEM OBSERVABILITY STATUS: {system_status}")
    print("=" * 65)
    
    if system_status == "CRITICAL ALERT":
        print("\nWARNING: Severe drift detected in live inference stream.")
        print("ACTION REQUIRED: Automated fallback engaged. Human review routing active.\n")
        
    # Using pandas internal display for clean table formatting in notebooks
    from IPython.display import display
    display(report_df)
    
    return report_df

# Execute the final pipeline report
final_report = generate_drift_report(reference_df, production_df)

      SYSTEM OBSERVABILITY STATUS: CRITICAL ALERT

ACTION REQUIRED: Automated fallback engaged. Human review routing active.



,Feature/Output,KS Stat,KS p-value,PSI,JS Divergence,Status
0,transaction_amount (Input Feature),0.4372,0.00e+00,0.9820,0.2115,🚨 SEVERE DRIFT
1,prediction_score (Model Output),0.6592,0.00e+00,2.8491,0.5506,🚨 SEVERE DRIFT
